In [1]:
import geopandas as gpd

In [2]:
gdf_kd = gpd.read_file('brez_filanja_lukenj.geojson')

In [3]:
gdf_kd.columns

Index(['ESD', 'EID', 'IME', 'SINONIMI', 'OPIS', 'ZVRST', 'TIP', 'GESLA',
       'DATACIJA', 'LOKACIJAOPIS', 'OBCINA', 'ZAVOD', 'USMERITVE', 'STATUS',
       'PREDPIS', 'PREDPISHTTP', 'VELJAVNOST', 'FOTOAVTOR', 'FOTODATOTEKA',
       'POVEZAVA1', 'SPOMENIK', 'OBCINAID', 'QR', 'X', 'Y', 'PHOTOURL',
       'poplave_ocena', 'poplave', 'pozar', 'plazovi', 'regija', 'geometry'],
      dtype='str')

In [4]:
gdf_po_upravah = gpd.read_file('Data/splosnoUE/splosno_nevarnosti.shp')

In [5]:
gdf_po_upravah.columns

Index(['ENOTA', 'UE_MID', 'UE_ID', 'UE_UIME', 'POV_KM2', 'D_OD', 'DV_OD',
       'STATUS', 'CEN_E', 'CEN_N', 'poplave', 'pozar', 'plazovi', 'potres',
       'geometry'],
      dtype='str')

In [6]:
gdf_po_upravah['STATUS'].unique()

<StringArray>
['V']
Length: 1, dtype: str

In [7]:
gdf_po_upravah.drop(columns=['ENOTA', 'UE_MID', 'UE_ID', 'D_OD', 'DV_OD', 'STATUS'])

,UE_UIME,POV_KM2,CEN_E,CEN_N,poplave,pozar,plazovi,potres,geometry
0,Ajdovščina,352.69,414738.0,83647.0,2,1,2,3,"POLYGON ((14.04966 45.83737, 14.04968 45.83735..."
1,Brežice,268.65,545949.0,85164.0,3,4,1,3,"POLYGON ((15.50385 45.86195, 15.50379 45.86215..."
2,Celje,230.02,520340.0,121185.0,2,3,2,1,"POLYGON ((15.33091 46.24124, 15.3309 46.24122,..."
3,Cerknica,482.86,450098.0,73146.0,2,4,1,2,"POLYGON ((14.27252 45.82223, 14.27254 45.82224..."
4,Črnomelj,488.56,515158.0,48225.0,1,4,1,3,"POLYGON ((15.22681 45.69842, 15.22679 45.69839..."
5,Domžale,239.57,469229.0,111866.0,2,3,1,2,"POLYGON ((14.6236 46.09621, 14.62356 46.09621,..."
6,Dravograd,104.94,502141.0,161466.0,2,4,3,1,"POLYGON ((15.06875 46.53416, 15.06871 46.53413..."
7,Gornja Radgona,213.49,575942.0,171103.0,1,4,1,1,"POLYGON ((15.96711 46.55834, 15.96711 46.55854..."
8,Grosuplje,458.33,473349.0,90646.0,1,4,1,2,"POLYGON ((14.86495 45.98437, 14.8653 45.98418,..."
9,Hrastnik,58.54,506650.0,112015.0,3,2,3,1,"POLYGON ((15.14778 46.08862, 15.14722 46.08837..."


In [8]:
gdf_kd = gdf_kd.sjoin(gdf_po_upravah[['geometry', 'UE_UIME']], how='left', predicate='within')

In [9]:
gdf_kd = gdf_kd.drop(columns=['index_right'])

In [10]:
gdf_kd.loc[gdf_kd['UE_UIME'].isna(), 'pozar'] = 1   
gdf_kd.loc[gdf_kd['UE_UIME'].isna(), 'plazovi'] = 0   
gdf_kd.loc[gdf_kd['UE_UIME'].isna(), 'UE_UIME'] = 'Koper'

In [11]:
ue_lookup = gdf_po_upravah.set_index('UE_UIME')
gdf_kd['potres'] = 0

for col in ['poplave', 'pozar', 'plazovi', 'potres']:
    mask = gdf_kd[col].isna()
    gdf_kd.loc[mask, col] = gdf_kd.loc[mask, 'UE_UIME'].map(ue_lookup[col])

In [12]:
#gdf_kd = gdf_kd.drop(columns=['poplave_ocena'])
gdf_kd.isna().sum()


ESD                  0
EID                  0
IME                  0
SINONIMI         19977
OPIS                 0
ZVRST                0
TIP                282
GESLA                0
DATACIJA           257
LOKACIJAOPIS         0
OBCINA               0
ZAVOD                0
USMERITVE         1401
STATUS               0
PREDPIS          22645
PREDPISHTTP      22645
VELJAVNOST       31086
FOTOAVTOR        12960
FOTODATOTEKA         0
POVEZAVA1            0
SPOMENIK         22645
OBCINAID         31086
QR                   0
X                    0
Y                    0
PHOTOURL             0
poplave_ocena    31086
poplave              0
pozar                0
plazovi              0
regija               0
geometry             0
UE_UIME              0
potres               0
dtype: int64

In [20]:
gdf_kd.to_file('kd_z_nevarnost.geojson', driver='GeoJSON')